In [0]:
%pip install shapely

In [0]:
from pyspark.sql import functions as F

CATALOG= "hawkerreach"
raw= spark.table(f"{CATALOG}.bronze.raw_hawker_centres")
display(raw.groupBy("status").count())

In [0]:
KEEP_STATUS= ["Existing", "Existing (new)", "Existing (replacement)", "Interim Centre"]

centres= (raw
    .filter(F.col("status").isin(KEEP_STATUS))
    .withColumn("cooked_food_stalls", F.col("number_of_cooked_food_stalls").cast("int"))
    .filter(F.col("cooked_food_stalls") > 0)  # drops wet-market-only rows
    .withColumn("lon", F.get_json_object("geometry", "$.coordinates[0]").cast("double"))
    .withColumn("lat", F.get_json_object("geometry", "$.coordinates[1]").cast("double"))
    .withColumn("opening_year", F.regexp_extract("est_original_completion_date", r"^(\d{4})", 1))
    .withColumn("opening_year", F.when(F.col("opening_year") == "", None).otherwise(F.col("opening_year").cast("int")))
    .withColumn("is_pre_war", F.coalesce(F.lower("est_original_completion_date").contains("pre-war"), F.lit(False)))
    .withColumn("is_interim", F.col("status") == "Interim Centre")
    .withColumn("centre_id", F.concat(F.lit("HC_"), F.col("objectid")))
    .select("centre_id", "name", "status", "is_interim", "cooked_food_stalls", "lon", "lat",
            "opening_year", "is_pre_war", "description", "addressblockhousenumber",
            "addressstreetname", "addresspostalcode", "info_on_co_locators")
)

centres.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.centres")
print("centres:", centres.count())

In [0]:
c= spark.table(f"{CATALOG}.silver.centres")
print("duplicate ids:", c.count() - c.select("centre_id").distinct().count())
print("missing coords:", c.filter(F.col("lon").isNull() | F.col("lat").isNull()).count())
print("outside Singapore:", c.filter(~F.col("lon").between(103.6, 104.1) | ~F.col("lat").between(1.2, 1.48)).count())
display(c.filter(F.col("opening_year").isNull()).select("name", "is_pre_war"))

In [0]:
import json
from collections import Counter, defaultdict
import pandas as pd
from shapely.geometry import shape

MIN_JACCARD= 0.8

bld= spark.table(f"{CATALOG}.bronze.raw_hdb_buildings").select("blk_no", "st_cod", "postal_cod", "geometry").toPandas()
prop= spark.table(f"{CATALOG}.bronze.raw_hdb_property").drop("source_url", "downloaded_at", "ingested_at").toPandas()

bld["blk_no"]= bld["blk_no"].str.strip().str.upper()
prop["blk_no"]= prop["blk_no"].str.strip().str.upper()
prop["street"]= prop["street"].str.strip().str.upper()

# one point per building, guaranteed inside the footprint (a centroid can fall outside an L-shaped block)
pts= bld["geometry"].map(lambda g: shape(json.loads(g)).representative_point())
bld["lon"]= pts.map(lambda p: p.x)
bld["lat"]= pts.map(lambda p: p.y)

# score every candidate street per code by Jaccard overlap of block numbers; keep all tied for best
code_blks= bld.groupby("st_cod")["blk_no"].apply(set)
street_blks= prop.groupby("street")["blk_no"].apply(set)
blk_to_streets= defaultdict(set)
for street, blks in street_blks.items():
    for b in blks:
        blk_to_streets[b].add(street)

cands= []
for code, blks in code_blks.items():
    overlap= Counter(s for b in blks for s in blk_to_streets.get(b, ()))
    if not overlap:
        continue
    scores= {s: inter / (len(blks) + len(street_blks[s]) - inter) for s, inter in overlap.items()}
    best_j= max(scores.values())
    cands.append((code, [s for s, j in scores.items() if j == best_j], best_j))

# break ties by location: pick the street whose town is nearest to the code's buildings
street_town= prop.groupby("street")["bldg_contract_town"].agg(lambda s: s.mode()[0])
code_xy= bld.groupby("st_cod")[["lon", "lat"]].mean()
clear= {code: tied[0] for code, tied, j in cands if len(tied) == 1 and j >= MIN_JACCARD}
town_xy= (code_xy.loc[list(clear)]
          .assign(town=[street_town[clear[c]] for c in clear])
          .groupby("town")[["lon", "lat"]].mean())

def nearest_street(code, tied):
    x, y= code_xy.loc[code, "lon"], code_xy.loc[code, "lat"]
    def dist(s):
        t= street_town[s]
        if t not in town_xy.index:
            return float("inf")
        return (town_xy.loc[t, "lon"] - x)**2 + (town_xy.loc[t, "lat"] - y)**2
    return min(tied, key=dist)

rows= []
for code, tied, j in cands:
    street= tied[0] if len(tied) == 1 else nearest_street(code, tied)
    rows.append((code, street, j, len(tied)))
code_map= pd.DataFrame(rows, columns=["st_cod", "street", "jaccard", "n_tied"])

good= code_map[code_map["jaccard"] >= MIN_JACCARD]
blocks= bld.merge(good[["st_cod", "street"]], on="st_cod").merge(prop, on=["blk_no", "street"])
res= prop[prop["residential"] == "Y"]
blocks_res= blocks[blocks["residential"] == "Y"]

print("street codes:", len(code_map), "| matched at >=", MIN_JACCARD, ":", len(good), "| ties resolved by location:", (good["n_tied"] > 1).sum())
print("residential blocks:", len(res), "| matched to a building:", len(blocks_res))
print("duplicate blk+street:", blocks_res.duplicated(["blk_no", "street"]).sum())
display(good[good["n_tied"] > 1].merge(bld.groupby("st_cod")["postal_cod"].first().reset_index(), on="st_cod"))

In [0]:
# a property row matched by two buildings: keep the one nearest its street's town, drop the other
blocks_res= blocks_res.copy()
blocks_res["town_lon"]= blocks_res["street"].map(street_town).map(town_xy["lon"])
blocks_res["town_lat"]= blocks_res["street"].map(street_town).map(town_xy["lat"])
blocks_res["d_town"]= (blocks_res["lon"] - blocks_res["town_lon"])**2 + (blocks_res["lat"] - blocks_res["town_lat"])**2
dropped= blocks_res[blocks_res.duplicated(["blk_no", "street"], keep=False)].sort_values("d_town").iloc[1:]
blocks_res= (blocks_res.sort_values("d_town")
             .drop_duplicates(["blk_no", "street"])
             .drop(columns=["town_lon", "town_lat", "d_town"]))
if len(dropped):
    display(dropped[["blk_no", "street", "st_cod", "postal_cod", "lon", "lat"]])
print("duplicate blk+street:", blocks_res.duplicated(["blk_no", "street"]).sum(), "| blocks:", len(blocks_res))

In [0]:
NUM_COLS= ["max_floor_lvl", "year_completed", "total_dwelling_units",
           "1room_sold", "2room_sold", "3room_sold", "4room_sold", "5room_sold", "exec_sold",
           "multigen_sold", "studio_apartment_sold",
           "1room_rental", "2room_rental", "3room_rental", "other_room_rental"]

out= blocks_res.copy()
for c in NUM_COLS:
    out[c]= out[c].astype(int)
out["block_id"]= "B_" + out["postal_cod"].str.strip()
out= out[["block_id", "blk_no", "street", "st_cod", "postal_cod", "bldg_contract_town", "lon", "lat"]
         + NUM_COLS + ["commercial", "market_hawker", "multistorey_carpark"]]

spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG}.silver.blocks")
print("blocks written:", len(out))

In [0]:
b= spark.table(f"{CATALOG}.silver.blocks")
print("rows:", b.count(), "| distinct block_id:", b.select("block_id").distinct().count())
print("outside Singapore:", b.filter(~F.col("lon").between(103.6, 104.1) | ~F.col("lat").between(1.2, 1.48)).count())
print("total dwelling units:", b.agg(F.sum("total_dwelling_units")).first()[0])